In [1]:
#install clip
!pip install ftfy regex tqdm
!pip install openai_clip

In [2]:
#necessary imports
import torch
import torchvision
import torch.nn as nn
import clip
from torch.nn import functional as F
from tqdm import tqdm
from collections import OrderedDict
import contextlib



/opt/conda/lib/python3.12/site-packages/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


We will now create functions to correctly get our data and split it into base and novel classes.

In [3]:
def get_data(data_dir="./data", transform=None):
    """Load Flowers102 train, validation and test sets.
    Args:
        data_dir (str): Directory where the dataset will be stored.
        transform (torch.Compose)
    Returns:
        tuple: A tuple containing the train, validation, and test sets.
    """
    train = torchvision.datasets.Flowers102(root=data_dir, split="train", download=True, transform=transform)
    val = torchvision.datasets.Flowers102(root=data_dir, split="val", download=True, transform=transform)
    test = torchvision.datasets.Flowers102(root=data_dir, split="test", download=True, transform=transform)
    return train, val, test

In [4]:
def base_novel_categories(dataset):
    # set returns the unique set of all dataset classes
    all_classes = set(dataset._labels)
    # and let's count them
    num_classes = len(all_classes)

    # here list(range(num_classes)) returns a list from 0 to num_classes - 1
    # then we slice the list in half and generate base and novel category lists
    base_classes = list(range(num_classes))[:num_classes//2]
    novel_classes = list(range(num_classes))[num_classes//2:]
    return base_classes, novel_classes

Let's inspect the classes.

In [5]:
_, _, tmp_test = get_data()
base_classes, novel_classes = base_novel_categories(tmp_test)
CLASS_NAMES = ["pink primrose", "hard-leaved pocket orchid", "canterbury bells", "sweet pea",
                "english marigold", "tiger lily", "moon orchid", "bird of paradise", "monkshood",
                "globe thistle", "snapdragon", "colt's foot", "king protea", "spear thistle",
                "yellow iris", "globe-flower", "purple coneflower", "peruvian lily", "balloon flower",
                "giant white arum lily", "fire lily", "pincushion flower", "fritillary", "red ginger",
                "grape hyacinth", "corn poppy", "prince of wales feathers", "stemless gentian", "artichoke",
                "sweet william", "carnation", "garden phlox", "love in the mist", "mexican aster",
                "alpine sea holly", "ruby-lipped cattleya", "cape flower", "great masterwort", "siam tulip",
                "lenten rose", "barbeton daisy", "daffodil", "sword lily", "poinsettia", "bolero deep blue",
                "wallflower", "marigold", "buttercup", "oxeye daisy", "common dandelion", "petunia", "wild pansy",
                "primula", "sunflower", "pelargonium", "bishop of llandaff", "gaura", "geranium", "orange dahlia",
                "pink-yellow dahlia", "cautleya spicata", "japanese anemone", "black-eyed susan", "silverbush",
                "californian poppy", "osteospermum", "spring crocus", "bearded iris", "windflower", "tree poppy",
                "gazania", "azalea", "water lily", "rose", "thorn apple", "morning glory", "passion flower", "lotus",
                "toad lily", "anthurium", "frangipani", "clematis", "hibiscus", "columbine", "desert-rose",
                "tree mallow", "magnolia", "cyclamen", "watercress", "canna lily", "hippeastrum", "bee balm",
                "ball moss", "foxglove", "bougainvillea", "camellia", "mallow", "mexican petunia", "bromelia",
                "blanket flower", "trumpet creeper", "blackberry lily"]
print("Base Class Names:", [(i, CLASS_NAMES[i]) for i in base_classes])
print("Novel Class Names:", [(i, CLASS_NAMES[i]) for i in novel_classes])

Base Class Names: [(0, 'pink primrose'), (1, 'hard-leaved pocket orchid'), (2, 'canterbury bells'), (3, 'sweet pea'), (4, 'english marigold'), (5, 'tiger lily'), (6, 'moon orchid'), (7, 'bird of paradise'), (8, 'monkshood'), (9, 'globe thistle'), (10, 'snapdragon'), (11, "colt's foot"), (12, 'king protea'), (13, 'spear thistle'), (14, 'yellow iris'), (15, 'globe-flower'), (16, 'purple coneflower'), (17, 'peruvian lily'), (18, 'balloon flower'), (19, 'giant white arum lily'), (20, 'fire lily'), (21, 'pincushion flower'), (22, 'fritillary'), (23, 'red ginger'), (24, 'grape hyacinth'), (25, 'corn poppy'), (26, 'prince of wales feathers'), (27, 'stemless gentian'), (28, 'artichoke'), (29, 'sweet william'), (30, 'carnation'), (31, 'garden phlox'), (32, 'love in the mist'), (33, 'mexican aster'), (34, 'alpine sea holly'), (35, 'ruby-lipped cattleya'), (36, 'cape flower'), (37, 'great masterwort'), (38, 'siam tulip'), (39, 'lenten rose'), (40, 'barbeton daisy'), (41, 'daffodil'), (42, 'sword 

Let's now split the dataset.

In [6]:
def split_data(dataset, base_classes):
    # these two lists will store the sample indexes
    base_categories_samples = []
    novel_categories_samples = []

    # we create a set of base classes to compute the test below in O(1)
    # this is optional and can be removed
    base_set = set(base_classes)

    # here we iterate over sample labels and also get the correspondent sample index
    for sample_id, label in enumerate(dataset._labels):
        if label in base_set:
            base_categories_samples.append(sample_id)
        else:
            novel_categories_samples.append(sample_id)

    # here we create the dataset subsets
    # the torch Subset is just a wrapper around the dataset
    # it simply stores the subset indexes and the original dataset (your_subset.dataset)
    # when asking for sample i in the subset, torch will look for its original position in the dataset and retrieve it
    # https://pytorch.org/docs/stable/data.html#torch.utils.data.Subset
    base_dataset = torch.utils.data.Subset(dataset, base_categories_samples)
    novel_dataset = torch.utils.data.Subset(dataset, novel_categories_samples)
    return base_dataset, novel_dataset

In [7]:
def create_remapped_dataset(dataset, selected_classes):
    """Create a dataset subset with remapped labels.

    Args:
        dataset: Original dataset
        selected_classes: List of class indices to include

    Returns:
        Subset dataset with labels remapped to [0, len(selected_classes)-1]
    """
    # Create mapping from original labels to new labels
    label_map = {old_label: new_label for new_label, old_label in enumerate(selected_classes)}
    selected_set = set(selected_classes)

    # Find samples and create new labels
    selected_samples = []
    new_labels = []

    for sample_id, label in enumerate(dataset._labels):
        if label in selected_set:
            selected_samples.append(sample_id)
            new_labels.append(label_map[label])

    # Create subset
    subset = torch.utils.data.Subset(dataset, selected_samples)

    # Add remapped labels to subset
    subset.remapped_labels = new_labels

    return subset

class RemappedDataset(torch.utils.data.Dataset):
    #TODO: vibecodato, funziona
    """Wrapper dataset that returns remapped labels"""
    def __init__(self, subset_dataset):
        self.dataset = subset_dataset.dataset
        self.indices = subset_dataset.indices
        self.labels = subset_dataset.remapped_labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # Get original sample
        original_idx = self.indices[idx]
        image, _ = self.dataset[original_idx]  # Ignore original label

        # Return with remapped label
        return image, self.labels[idx]

Let's now load a pretrained CLIP model.

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

# Load the CLIP model and preprocessing transform
clip_model, preprocess = clip.load("ViT-B/16", device=device)
clip_model.eval()  # We won't fine-tune CLIP
for param in clip_model.parameters():
    param.requires_grad = False

# Get the image and text encoders
image_encoder = clip_model.visual
text_encoder = clip_model.encode_text  # Used in inference mode only


cuda


Let's prepare the train-test-splits.


In [9]:
# get the three datasets
train_set, val_set, test_set = get_data(transform=preprocess)

# split classes into base and novel
base_classes, novel_classes = base_novel_categories(train_set)

# split the three datasets
train_base, _ = split_data(train_set, base_classes)
val_base, _ = split_data(val_set, base_classes)
test_base, test_novel = split_data(test_set, base_classes)

###### Now we will start implementing CoOp.

 The first thing to do is create a **PromptLearner** class. We will randomly initialize the context vector and **class_name** will be at the end of the prompt followed by EOS token. We will use 32 as number of context tokens.

In [10]:
class PromptLearner(nn.Module):
    def __init__(self, active_class_indices, all_class_names, clip_model, n_ctx=4):
      super().__init__()
      self.all_class_names = all_class_names
      self.n_all_cls = len(all_class_names)
      self.active_class_names = active_class_indices
      self.n_active_cls = len(active_class_indices)
      self.n_ctx = n_ctx
      self.clip_model = clip_model
      self.tokenizer = clip.tokenize

      # CLIP parameters
      self.dtype = clip_model.dtype
      ctx_dim = clip_model.ln_final.weight.shape[0] # i.e. 512
      self.ctx_dim = ctx_dim
      # Get the device from the clip_model
      self.device = next(clip_model.parameters()).device


      # Random context initialization
      ctx_vectors = torch.empty(n_ctx, ctx_dim, dtype=self.dtype)
      nn.init.normal_(ctx_vectors, std=0.02)
      prompt_prefix = " ".join(["X"] * n_ctx)

      self.ctx = nn.Parameter(ctx_vectors)  # shared learnable context, to be optimized

      
      # templates with placeholders. we use all class names (base + novel)
      prompt_prefix = " ".join(["X"] * n_ctx)
      template_prompts = [f"{prompt_prefix} {name}." for name in all_class_names]
      tokenized_template_prompts = torch.cat([clip.tokenize(p) for p in template_prompts]).to(self.device) # [all_C, 77]
      with torch.no_grad():
          embedded_template_prompts = clip_model.token_embedding(tokenized_template_prompts).type(self.dtype) # [all_C, 77, 512]

      self.register_buffer("token_prefix", embedded_template_prompts[:, :1, :]) # SOT, [all_C, 1, 512]
      self.register_buffer("token_suffix", embedded_template_prompts[:, 1 + n_ctx :, :]) # CLS, EOS, [all_C, *, 512]

      self.tokenized_prompts = tokenized_template_prompts
      
    
    def set_active_classes(self, active_class_indices):
        """Update which classes are active for classification"""
        self.active_class_indices = active_class_indices
        self.active_C = len(active_class_indices)
        self.active_class_names = [self.all_class_names[i] for i in active_class_indices]

    
    # this function returns 102 tokenized sequences (same current learned ctx + each class name) 
    def forward(self, shift=None):

      prefix = self.token_prefix
      ctx = self.ctx.unsqueeze(0) # [1, n_ctx, 512]
      suffix = self.token_suffix


      if shift is not None:
        # we assume batch size to be 1 so everything is easier
        assert shift.shape[0] == 1
        shift = shift.unsqueeze(1) # [1, n_ctx, 512]
        ctx = ctx + shift
      
      ctx = ctx.expand(self.n_all_cls, -1, -1) # [all_C, n_ctx, 512]
      prompts = torch.cat([prefix, ctx, suffix], dim=1) #[all_C, 77, 512]
      
      return prompts

In [11]:
# Prompt Shifter (Meta-Net)

class PromptShifter(nn.Module):
    def __init__(self, vis_dim, ctx_dim, hidden_dim=16, dtype=None, device=None):
        super().__init__()
        if hidden_dim:
            self.net = nn.Sequential(
                nn.Linear(vis_dim, hidden_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(p=0.5),
                nn.Linear(hidden_dim, ctx_dim)
            )
        else:
            self.net = nn.Linear(vis_dim, ctx_dim)

        if dtype is not None:
            self.net = self.net.type(dtype)

        if device is not None:
            self.net = self.net.to(device)

    def forward(self, image_feats):
        shift = self.net(image_feats) # [B, ctx_dim]
        shift = shift / (shift.norm(dim=-1, keepdim=True) + 1e-6) # normalize
        return shift


We now have to create a **TextEncoder** class.

In [12]:
class TextEncoder(nn.Module):
  def __init__(self, clip_model):
    super().__init__()
    self.transformer = clip_model.transformer
    self.positional_embedding = clip_model.positional_embedding
    self.ln_final = clip_model.ln_final
    self.text_projection = clip_model.text_projection  # This was missing!
    self.dtype = clip_model.dtype

  def forward(self, prompts, tokenized_prompts):
    x = prompts + self.positional_embedding.type(self.dtype)
    x = x.permute(1,0,2)  # [seq_len, n_cls, embed_dim]
    x = self.transformer(x)
    x = x.permute(1,0,2)  # [n_cls, seq_len, embed_dim]
    x = self.ln_final(x).type(self.dtype)

    # Take features from EOS embedding
    x = x[torch.arange(x.shape[0]), tokenized_prompts.argmax(dim=-1)] @ self.text_projection

    return x

We are now ready to create out **CustomCLIP** class.

In [13]:
class KgCoCoOp(nn.Module):
  def __init__(self, clip_model, active_class_indices, all_class_names,
              n_ctx=4,
              alpha = 1.0,
              use_shift=False,
              shifter_hidden=16
              ):
    super().__init__()
    self.prompt_learner = PromptLearner(active_class_indices, all_class_names, clip_model, n_ctx)
    self.tokenized_prompts = self.prompt_learner.tokenized_prompts
    self.text_encoder = TextEncoder(clip_model)
    self.image_encoder = clip_model.visual
    self.logit_scale = clip_model.logit_scale
    self.dtype = clip_model.dtype
    self.use_shift = use_shift
    self.ctx_dim = self.prompt_learner.ctx_dim
    self.all_class_names = all_class_names
    self.active_class_indices = active_class_indices
    self.clip_model = clip_model
    self.device = next(clip_model.parameters()).device
    self.targets = self._compute_targets()
    self.alpha = alpha
    self.prompt_shifter = PromptShifter(vis_dim=clip_model.visual.output_dim, ctx_dim=self.ctx_dim, hidden_dim=shifter_hidden,
                                        device=self.device, dtype=self.dtype)

  
  def _compute_targets(self):
    template = "A photo of a {}, a type of flower."
    prompts = [template.format(name) for name in self.all_class_names] # list of 102 prompts (102 is the number of base+novel classes)
    tokenized = clip.tokenize(prompts).to(self.device) # [102, 77]
    with torch.no_grad():
      feats = self.clip_model.encode_text(tokenized).type(self.dtype) # [102, 512]
    feats = feats / feats.norm(dim=1, keepdim=True)
    return feats
  
  
  def _compute_text_loss(self, soft_feats):
    cos_sim = torch.sum(soft_feats * self.targets, dim=1)
    avg_cos_sim = torch.mean(cos_sim)
    loss = 1 - avg_cos_sim
    return self.alpha * loss
  

  def forward(self, images):

    # Encode images
    with torch.no_grad():
      image_features = self.image_encoder(images.type(self.dtype))
    image_features = image_features / image_features.norm(dim=1, keepdim=True)

    raw_prompts = self.prompt_learner()

    if self.use_shift:
      shift = self.prompt_shifter(image_features)
      shifted_prompts = self.prompt_learner(shift)
      active_prompts = shifted_prompts
    else:
      active_prompts = raw_prompts

    
    with torch.no_grad() if not self.training else contextlib.nullcontext():
      raw_text_features = self.text_encoder(raw_prompts, self.tokenized_prompts)
      raw_text_features = raw_text_features / raw_text_features.norm(dim=1, keepdim=True)
      text_loss = self._compute_text_loss(raw_text_features)
    
    # Compute text features. these are for ALL classes
    text_features = self.text_encoder(active_prompts, self.tokenized_prompts)
    text_features = text_features / text_features.norm(dim=1, keepdim=True)

    

    # Compute similarity logits
    logit_scale = self.logit_scale.exp()
    # only take text feats for active class indices
    text_features = text_features[self.active_class_indices]
    logits = logit_scale * image_features @ text_features.t() 

    return logits, text_loss

Let's now create training and evaluation functions.

In [14]:
def train_one_epoch(model, dataloader, optimizer, scheduler, loss_function, device):
  model.train()
  total_loss = 0.0
  total_correct = 0
  total_samples = 0

  for images, labels in dataloader:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()
    logits, text_loss = model(images)
    loss = loss_function(logits, labels) + text_loss
    loss.backward()
    optimizer.step()
    

    total_loss += loss.item() * images.size(0)
    total_correct += (logits.argmax(dim=1) == labels).sum().item()
    total_samples += images.size(0)

  scheduler.step()
  avg_loss = total_loss / total_samples
  accuracy = total_correct / total_samples
  return avg_loss, accuracy


def evaluate(model, dataloader, device):
  model.eval()
  total_correct = 0
  total_samples = 0

  with torch.no_grad():
    for images, labels in dataloader:
      images = images.to(device)
      labels = labels.to(device)

      logits, _ = model(images)
      total_correct += (logits.argmax(dim=1) == labels).sum().item()
      total_samples += images.size(0)

  accuracy = total_correct / total_samples
  return accuracy

Now let's set up the training for CoOp on the base classes.

In [15]:
# Create properly remapped datasets for base classes
train_base_remapped = create_remapped_dataset(train_set, base_classes)
val_base_remapped = create_remapped_dataset(val_set, base_classes)
test_base_remapped = create_remapped_dataset(test_set, base_classes)

# Wrap with RemappedDataset to get correct labels
train_base_dataset = RemappedDataset(train_base_remapped)
val_base_dataset = RemappedDataset(val_base_remapped)
test_base_dataset = RemappedDataset(test_base_remapped)

print(f"Base training samples: {len(train_base_dataset)}")
print(f"Base validation samples: {len(val_base_dataset)}")
print(f"Base test samples: {len(test_base_dataset)}")
print(f"Number of base classes: {len(base_classes)}")

# Get base class names for the model
base_class_names = [CLASS_NAMES[i] for i in base_classes]
print(f"Base class names: {base_class_names[:5]}...")  # Show first 5

Base training samples: 510
Base validation samples: 510
Base test samples: 2473
Number of base classes: 51
Base class names: ['pink primrose', 'hard-leaved pocket orchid', 'canterbury bells', 'sweet pea', 'english marigold']...


In [16]:
# Create data loaders
# For stage 1, we can use a larger batch size
# We will set it to 1 for stage 2
batch_size = 32
train_loader = torch.utils.data.DataLoader(
    train_base_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4
)
val_loader = torch.utils.data.DataLoader(
    val_base_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4
)
test_loader = torch.utils.data.DataLoader(
    test_base_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Training batches: 16
Validation batches: 16
Test batches: 78


In [17]:
n_ctx = 4
alpha = 1.0
shifter_hidden = 16

# Initialize the KgCoCoOp model
kgcocoop_model = KgCoCoOp(clip_model, base_classes, CLASS_NAMES,
                          n_ctx= n_ctx,
                          alpha= alpha,
                          use_shift=False,
                          shifter_hidden=shifter_hidden
                          ).to(device)

# ======== STAGE 1: prompt learning ==========
print("Starting stage 1: prompt learning (no MetaNet)")

# Only train ctx
for p in kgcocoop_model.parameters():
    p.requires_grad = False
kgcocoop_model.prompt_learner.ctx.requires_grad = True

# Loss function
loss_function = nn.CrossEntropyLoss()

print(f"Model initialized with {len(base_class_names)} base classes")
print(f"Context dimension: {kgcocoop_model.prompt_learner.ctx_dim}")
print(f"Number of context tokens: {kgcocoop_model.prompt_learner.n_ctx}")
print(f"Trainable parameters: {sum(p.numel() for p in kgcocoop_model.parameters() if p.requires_grad)}")

# Verify only context vectors are trainable
print("\nTrainable parameters:")
for name, param in kgcocoop_model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.shape}")

Starting stage 1: prompt learning (no MetaNet)
Model initialized with 51 base classes
Context dimension: 512
Number of context tokens: 4
Trainable parameters: 2048

Trainable parameters:
  prompt_learner.ctx: torch.Size([4, 512])


In [ ]:
optimizer = torch.optim.SGD([kgcocoop_model.prompt_learner.ctx], lr=0.002, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200, eta_min=0.0001)
num_epochs = 5
best_val_acc = 0.0


# Training loop
print("Starting training...")
training_history = {
    'train_loss': [],
    'train_acc': [],
    'val_acc': []
}

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 50)

    # Training phase
    train_loss, train_acc = train_one_epoch(kgcocoop_model, train_loader, optimizer, scheduler, loss_function, device)

    # Validation phase
    val_acc = evaluate(kgcocoop_model, val_loader, device)

    # Save training history
    training_history['train_loss'].append(train_loss)
    training_history['train_acc'].append(train_acc)
    training_history['val_acc'].append(val_acc)

    # Print metrics
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        print(f"New best validation accuracy: {best_val_acc:.4f}")
        # Save model checkpoint
        torch.save({
            'epoch': epoch,
            'model_state_dict': kgcocoop_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
        }, 'best_kgcocoop_model_stage1.pth')

print(f"\nTraining completed!")
print(f"Best validation accuracy: {best_val_acc:.4f}")

Starting training...

Epoch 1/5
--------------------------------------------------


In [ ]:
# Load best model and evaluate on test set
checkpoint = torch.load('best_kgcocoop_model_stage1.pth')
kgcocoop_model.load_state_dict(checkpoint['model_state_dict'])

test_acc = evaluate(kgcocoop_model, test_loader, device)
print(f"Test Accuracy on Base Classes: {test_acc:.4f}")

# Plot training curves
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(training_history['train_loss'], label='Training Loss')
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(training_history['train_acc'], label='Training Accuracy')
plt.plot(training_history['val_acc'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"\nFinal Results:")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
# Now let's evaluate on novel classes to test generalization
print("\n" + "="*60)
print("EVALUATING ON NOVEL CLASSES")
print("="*60)

# Create novel class datasets with remapped labels
test_novel_remapped = create_remapped_dataset(test_set, novel_classes)
test_novel_dataset = RemappedDataset(test_novel_remapped)

# Get novel class names
novel_class_names = [CLASS_NAMES[i] for i in novel_classes]
print(f"Novel classes: {len(novel_classes)}")
print(f"Novel test samples: {len(test_novel_dataset)}")
print(f"Novel class names: {novel_class_names[:5]}...")

# Create a new model for novel classes (same learned context, different class names)
novel_kgcocoop_model = KgCoCoOp(clip_model, novel_classes, CLASS_NAMES,
                            n_ctx=n_ctx,
                            alpha=alpha,
                            use_shift=False,
                            shifter_hidden=shifter_hidden
                            ).to(device)

# Load the learned context from the trained model
novel_kgcocoop_model.prompt_learner.ctx.data = kgcocoop_model.prompt_learner.ctx.data.clone()

# Create data loader for novel classes
test_novel_loader = torch.utils.data.DataLoader(
    test_novel_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

# Evaluate on novel classes
novel_test_acc = evaluate(novel_kgcocoop_model, test_novel_loader, device)
print(f"\nTest Accuracy on Novel Classes: {novel_test_acc:.4f}")

# Compare base vs novel performance
print(f"\n" + "="*60)
print("Stage 1 Summary")
print("="*60)
print(f"Base Classes Test Accuracy: {test_acc:.4f}")
print(f"Novel Classes Test Accuracy: {novel_test_acc:.4f}")
print(f"Harmonic mean: {(2 / (1/test_acc + 1/novel_test_acc)):.4f}")

# This shows how well the learned context generalizes to unseen classes

In [ ]:
# ========================
# ======== STAGE 2 =======
# ========================
print("\n" + "="*60)
print("Starting Stage 2: MetaNet training (prompt frozen)")
print("="*60)

# Load best prompt-tuned model from Stage 1
checkpoint = torch.load('best_kgcocoop_model_stage1.pth')
kgcocoop_model.load_state_dict(checkpoint['model_state_dict'])

# Enable MetaNet
kgcocoop_model.use_shift = True

# Freeze prompt context
# kgcocoop_model.prompt_learner.ctx.requires_grad = False

# Enable only PromptShifter
for p in kgcocoop_model.prompt_shifter.parameters():
    p.requires_grad = True

# Define a new optimizer and scheduler for PromptShifter
optimizer = torch.optim.SGD([
    {"params": kgcocoop_model.prompt_learner.ctx, "lr": 0.002},
    {"params": kgcocoop_model.prompt_shifter.parameters(), "lr": 0.004}
], momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

# Change batch size to 1 (per-image shifting)
train_loader_stage2 = torch.utils.data.DataLoader(
    train_base_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=2
)
val_loader_stage2 = torch.utils.data.DataLoader(
    val_base_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2
)
test_loader_stage2 = torch.utils.data.DataLoader(
    test_base_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2
)

# Define a clean training loop (without text supervision)
def train_stage2(model, dataloader, optimizer, scheduler, loss_function, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits, text_loss = model(images)  # Ignore text_loss in Stage 2
        loss = loss_function(logits, labels) + text_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += images.size(0)

    scheduler.step()
    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    return avg_loss, accuracy


In [ ]:
num_epochs_stage2 = 3
best_val_acc_stage2 = 0.0
training_history_stage2 = {
    'train_loss': [],
    'train_acc': [],
    'val_acc': []
}

for epoch in range(num_epochs_stage2):
    print(f"\nStage 2 — Epoch {epoch+1}/{num_epochs_stage2}")
    print("-" * 50)

    train_loss, train_acc = train_stage2(kgcocoop_model, train_loader_stage2, optimizer, scheduler, loss_function, device)
    val_acc = evaluate(kgcocoop_model, val_loader_stage2, device)

    training_history_stage2['train_loss'].append(train_loss)
    training_history_stage2['train_acc'].append(train_acc)
    training_history_stage2['val_acc'].append(val_acc)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")

    if val_acc > best_val_acc_stage2:
        best_val_acc_stage2 = val_acc
        print(f"New best validation accuracy (Stage 2): {best_val_acc_stage2:.4f}")
        torch.save({
            'epoch': epoch,
            'model_state_dict': kgcocoop_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
        }, 'best_kgcocoop_model_stage2.pth')


In [ ]:
print("\n" + "="*60)
print("EVALUATING STAGE 2 MODEL ON BASE AND NOVEL CLASSES")
print("="*60)

# Load best MetaNet model
checkpoint = torch.load('best_kgcocoop_model_stage2.pth')
kgcocoop_model.load_state_dict(checkpoint['model_state_dict'])
kgcocoop_model.use_shift = True

# Evaluate on base test set (MetaNet + learned ctx)
test_acc_base_stage2 = evaluate(kgcocoop_model, test_loader_stage2, device)
print(f"Test Accuracy on Base Classes (Stage 2): {test_acc_base_stage2:.4f}")

# Evaluate on novel classes
novel_model_stage2 = KgCoCoOp(
    clip_model=clip_model,
    active_class_indices=novel_classes,
    all_class_names=CLASS_NAMES,
    n_ctx=n_ctx,
    alpha=alpha,
    use_shift=True,
    shifter_hidden=shifter_hidden
).to(device)

# Load learned context and MetaNet weights
novel_model_stage2.prompt_learner.ctx.data = kgcocoop_model.prompt_learner.ctx.data.clone()
novel_model_stage2.prompt_shifter.load_state_dict(kgcocoop_model.prompt_shifter.state_dict())

# Use batch_size=1 for evaluation (MetaNet requires per-image shift)
test_novel_loader_stage2 = torch.utils.data.DataLoader(
    test_novel_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2
)

test_acc_novel_stage2 = evaluate(novel_model_stage2, test_novel_loader_stage2, device)
print(f"Test Accuracy on Novel Classes (Stage 2): {test_acc_novel_stage2:.4f}")

# Compute harmonic mean
hmean_stage2 = 2 / (1/test_acc_base_stage2 + 1/test_acc_novel_stage2)
print(f"Harmonic Mean (Stage 2): {hmean_stage2:.4f}")
